![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/llama.cpp/llama_cpp_in_Spark_NLP_LLMEntityExtractor.ipynb)

# LLM-based Entity Extraction with LLMEntityExtractor in Spark NLP 🚀

This notebook demonstrates `LLMEntityExtractor`, an end-to-end annotator for extracting entities from text using Large Language Models (LLMs)

**Key Features:**
- Uses AutoGGUF models (llama.cpp) for inference
- Constrained generation via BNF grammars ensures valid JSON output every time
- Few-shot prompting to guide the model with examples
- Outputs `CHUNK` annotations with accurate character-level begin/end indices
- Batch processing for high throughput
- Configurable entity types, confidence thresholds, and prompts

**How it works:**  
LLMEntityExtractor follows the **LangExtract** pattern from Google Research, combining few-shot prompting with constrained generation through llama.cpp BNF grammars. The LLM generates structured JSON, and the annotator performs string matching to locate each entity in the original text with precise character offsets.

## 1. Setup

Install and start Spark NLP. If running on Google Colab, use the setup script.

In [ ]:
# Only run this cell if you are on Google Colab
! wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [2]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# Start Spark with GPU support. Remove gpu=True if no GPU is available.
spark = sparknlp.start(gpu=True)

print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)

Spark NLP version: 6.4.0
Apache Spark version: 3.4.4


## 2. Basic Usage: Zero-Shot Entity Extraction

The simplest way to use `LLMEntityExtractor` is with zero-shot extraction. Just specify the entity types you want to extract and let the LLM do the rest. By default, the annotator uses the `qwen3_4b_bf16_gguf`.

In [13]:
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

entity_extractor = LLMEntityExtractor() \
    .pretrained("qwen3_4b_bf16_gguf") \
    .setInputCols(["document"]) \
    .setOutputCol("entities") \
    .setEntityTypes(["PERSON", "ORGANIZATION", "LOCATION", "DATE"]) \
    .setNPredict(500) \
    .setNGpuLayers(99) \
    .setTemperature(0.1) \
    .setBatchSize(4)

pipeline = Pipeline(stages=[document_assembler, entity_extractor])

qwen3_4b_bf16_gguf download started this may take some time.
Approximate size to download 6 GB
[OK!]


In [14]:
data = spark.createDataFrame([
    ["John Smith visited the United Nations headquarters in New York on January 15th, 2024."],
    ["Dr. Emily Chen from Stanford University presented her research at the WHO conference in Geneva."],
    ["Apple Inc. CEO Tim Cook announced the new product line at their Cupertino campus on March 3rd."]
]).toDF("text")

result = pipeline.fit(data).transform(data)

In [15]:
# Show extracted entities
result.select("text", "entities.result", "entities.metadata").show(truncate=80)

+--------------------------------------------------------------------------------+----------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            text|                                                    result|                                                                        metadata|
+--------------------------------------------------------------------------------+----------------------------------------------------------+--------------------------------------------------------------------------------+
|John Smith visited the United Nations headquarters in New York on January 15t...|[John Smith, United Nations, New York, January 15th, 2024]|[{entity -> PERSON, chunk -> 0, sentence -> 0}, {entity -> ORGANIZATION, chun...|
|Dr. Emily Chen from Stanford University presented her research at the WHO con...|        [Dr. Emily Chen, S

In [16]:
# Explore the full annotation structure — entity type, chunk index, and character offsets
result.select(
    F.explode("entities").alias("entity")
).select(
    F.col("entity.result").alias("text"),
    F.col("entity.begin").alias("begin"),
    F.col("entity.end").alias("end"),
    F.col("entity.metadata.entity").alias("entity_type"),
    F.col("entity.metadata.chunk").alias("chunk_index")
).show(truncate=False)

+-------------------+-----+---+------------+-----------+
|text               |begin|end|entity_type |chunk_index|
+-------------------+-----+---+------------+-----------+
|John Smith         |0    |9  |PERSON      |0          |
|United Nations     |23   |36 |ORGANIZATION|1          |
|New York           |54   |61 |LOCATION    |2          |
|January 15th, 2024 |66   |83 |DATE        |3          |
|Dr. Emily Chen     |0    |13 |PERSON      |0          |
|Stanford University|20   |38 |ORGANIZATION|1          |
|Geneva             |88   |93 |LOCATION    |2          |
|WHO                |70   |72 |ORGANIZATION|3          |
|Apple Inc.         |0    |9  |ORGANIZATION|0          |
|Tim Cook           |15   |22 |PERSON      |1          |
|Cupertino          |64   |72 |LOCATION    |2          |
|March 3rd          |84   |92 |DATE        |3          |
+-------------------+-----+---+------------+-----------+



## 3. Few-Shot NER: Medical Entity Extraction

Few-shot examples significantly improve extraction quality. Each example is a tuple of `(input_text, expected_json_output)` that teaches the LLM the exact format and entity types you expect.

This is especially powerful for domain-specific NER (medical, legal, financial, etc.) where the LLM benefits from seeing concrete examples.

In [19]:
# Define few-shot examples to guide the model
medical_examples = [
    (
        "Patient takes aspirin 81mg daily.",
        '{"extractions": [{"entity": "MEDICATION", "text": "aspirin", "confidence": 0.95}, '
        '{"entity": "DOSAGE", "text": "81mg", "confidence": 0.98}, '
        '{"entity": "FREQUENCY", "text": "daily", "confidence": 0.99}]}'
    ),
    (
        "Dr. Michael Chen from Massachusetts General Hospital ordered a chest X-ray and "
        "prescribed lisinopril 10mg PO BID for patient Sarah Williams on March 15th, 2024.",
        '{"extractions": [{"entity": "PERSON", "text": "Dr. Michael Chen", "confidence": 0.95}, '
        '{"entity": "ORGANIZATION", "text": "Massachusetts General Hospital", "confidence": 0.98}, '
        '{"entity": "TEST", "text": "chest X-ray", "confidence": 0.97}, '
        '{"entity": "MEDICATION", "text": "lisinopril", "confidence": 0.96}, '
        '{"entity": "DOSAGE", "text": "10mg", "confidence": 0.98}, '
        '{"entity": "ROUTE", "text": "PO", "confidence": 0.97}, '
        '{"entity": "FREQUENCY", "text": "BID", "confidence": 0.99}, '
        '{"entity": "PERSON", "text": "Sarah Williams", "confidence": 0.94}]}'
    ),
    (
        "Patient Jennifer Thompson received vancomycin 1g IV Q12H at Cleveland Clinic "
        "for treatment of MRSA pneumonia diagnosed on January 20th, 2024.",
        '{"extractions": [{"entity": "PERSON", "text": "Jennifer Thompson", "confidence": 0.95}, '
        '{"entity": "MEDICATION", "text": "vancomycin", "confidence": 0.98}, '
        '{"entity": "DOSAGE", "text": "1g", "confidence": 0.99}, '
        '{"entity": "ROUTE", "text": "IV", "confidence": 0.97}, '
        '{"entity": "FREQUENCY", "text": "Q12H", "confidence": 0.97}, '
        '{"entity": "ORGANIZATION", "text": "Cleveland Clinic", "confidence": 0.96}, '
        '{"entity": "CONDITION", "text": "MRSA pneumonia", "confidence": 0.98}]}'
    )
]


In [20]:
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

medical_entity_extractor = LLMEntityExtractor() \
    .pretrained("qwen3_4b_bf16_gguf") \
    .setInputCols(["document"]) \
    .setOutputCol("entities") \
    .setEntityTypes([
        "PERSON", "ORGANIZATION", "LOCATION",
        "MEDICATION", "DOSAGE", "ROUTE", "FREQUENCY",
        "TEST", "CONDITION"
    ]) \
    .setFewShotExamples(medical_examples) \
    .setNPredict(600) \
    .setNGpuLayers(99) \
    .setTemperature(0.1) \
    .setNCtx(5120) \
    .setTopK(40) \
    .setTopP(0.9) \
    .setBatchSize(4)

medical_pipeline = Pipeline(stages=[document_assembler, medical_entity_extractor])

qwen3_4b_bf16_gguf download started this may take some time.
Approximate size to download 6 GB
[OK!]


In [21]:
medical_texts = [
    "Dr. Sarah Johnson from Stanford Medical Center prescribed 500mg amoxicillin PO TID to "
    "patient John Smith on January 15th, 2024 for acute bronchitis.",

    "The laboratory at Massachusetts General Hospital reported elevated troponin levels of "
    "0.8 ng/mL at 14:30 on March 3rd indicating possible myocardial infarction.",

    "Patient Mary Williams received 10mg morphine IV push at 08:00 for post-operative pain "
    "management following her appendectomy at Cleveland Clinic on February 20, 2024.",

    "Dr. Robert Chen ordered a complete metabolic panel and chest X-ray for patient David "
    "Rodriguez at Johns Hopkins Hospital due to suspected pneumonia and dehydration.",

    "The oncology department at Memorial Sloan Kettering Cancer Center administered 75mg/m2 "
    "cisplatin IV infusion to patient Emily Davis on April 10th as part of her chemotherapy regimen.",
]

medical_data = spark.createDataFrame([[text] for text in medical_texts]).toDF("text")
medical_result = medical_pipeline.fit(medical_data).transform(medical_data)

In [22]:
# Display the extracted medical entities with full detail
medical_result.select(
    F.explode("entities").alias("entity")
).select(
    F.col("entity.result").alias("extracted_text"),
    F.col("entity.metadata.entity").alias("entity_type"),
    F.col("entity.begin").alias("begin"),
    F.col("entity.end").alias("end"),
    F.col("entity.metadata.chunk").alias("chunk")
).show(50, truncate=False)

+--------------------------------------+------------+-----+---+-----+
|extracted_text                        |entity_type |begin|end|chunk|
+--------------------------------------+------------+-----+---+-----+
|Dr. Sarah Johnson                     |PERSON      |0    |16 |0    |
|Stanford Medical Center               |ORGANIZATION|23   |45 |1    |
|amoxicillin                           |MEDICATION  |64   |74 |2    |
|500mg                                 |DOSAGE      |58   |62 |3    |
|PO                                    |ROUTE       |76   |77 |4    |
|TID                                   |FREQUENCY   |79   |81 |5    |
|John Smith                            |PERSON      |94   |103|6    |
|acute bronchitis                      |CONDITION   |131  |146|7    |
|Massachusetts General Hospital        |ORGANIZATION|18   |47 |0    |
|troponin levels                       |TEST        |67   |81 |1    |
|0.8 ng/mL                             |DOSAGE      |86   |94 |2    |
|laboratory         

## 4. Tuning LLM Parameters

`LLMEntityExtractor` inherits all llama.cpp parameters from `HasLlamaCppProperties`. Here are the key ones for entity extraction performance:

| Parameter | Effect on entity extraction | Recommended |
|---|---|---|
| `temperature` | Lower = more deterministic extractions | `0.1` - `0.2` |
| `topK` | Limits token sampling pool | `40` |
| `topP` | Nucleus sampling threshold | `0.9` |
| `nPredict` | Max output tokens (must fit all entities) | `500` - `800` |
| `nCtx` | Context window (must fit prompt + text + output) | `4096` - `8192` |
| `nGpuLayers` | Layers offloaded to GPU (99 = all) | `99` if GPU available |
| `batchSize` | Documents processed in parallel | `4` - `8` |

In [23]:
# Example: Tuning for long documents with many entities
tuned_entity_extractor = LLMEntityExtractor() \
    .pretrained("qwen3_4b_bf16_gguf") \
    .setInputCols(["document"]) \
    .setOutputCol("entities") \
    .setEntityTypes(["PERSON", "ORGANIZATION", "LOCATION", "DATE", "MEDICATION", "DOSAGE"]) \
    .setFewShotExamples(medical_examples) \
    .setNPredict(800) \
    .setNCtx(8192) \
    .setNGpuLayers(99) \
    .setTemperature(0.15) \
    .setTopK(40) \
    .setTopP(0.9) \
    .setPenalizeNl(True) \
    .setBatchSize(4)

tuned_pipeline = Pipeline(stages=[document_assembler, tuned_entity_extractor])

long_text = spark.createDataFrame([
    ["Dr. Jennifer Lee from UCLA Medical Center prescribed metformin 1000mg PO BID and "
     "lisinopril 20mg PO daily for patient Thomas Anderson's type 2 diabetes and hypertension. "
     "The cardiology team at Cedars-Sinai Medical Center performed a transesophageal "
     "echocardiogram on patient Jessica Taylor at 10:30 revealing severe mitral valve "
     "regurgitation. Patient Andrew Thompson received vancomycin 1g IV Q12H and ceftriaxone "
     "2g IV daily at Northwestern Memorial Hospital for treatment of bacterial meningitis "
     "starting August 1st, 2024."]
]).toDF("text")

tuned_result = tuned_pipeline.fit(long_text).transform(long_text)

tuned_result.select(
    F.explode("entities").alias("entity")
).select(
    F.col("entity.result").alias("text"),
    F.col("entity.metadata.entity").alias("type"),
    F.col("entity.begin").alias("begin"),
    F.col("entity.end").alias("end")
).show(50, truncate=False)

qwen3_4b_bf16_gguf download started this may take some time.
Approximate size to download 6 GB
[OK!]
+------------------------------+------------+-----+---+
|text                          |type        |begin|end|
+------------------------------+------------+-----+---+
|Dr. Jennifer Lee              |PERSON      |0    |15 |
|UCLA Medical Center           |ORGANIZATION|22   |40 |
|metformin                     |MEDICATION  |53   |61 |
|1000mg PO BID                 |DOSAGE      |63   |75 |
|lisinopril                    |MEDICATION  |81   |90 |
|20mg PO daily                 |DOSAGE      |92   |104|
|Thomas Anderson               |PERSON      |118  |132|
|Cedars-Sinai Medical Center   |ORGANIZATION|193  |219|
|Jessica Taylor                |PERSON      |275  |288|
|Andrew Thompson               |PERSON      |352  |366|
|vancomycin                    |MEDICATION  |377  |386|
|1g IV Q12H                    |DOSAGE      |388  |397|
|ceftriaxone                   |MEDICATION  |403  |413|
|2g

## 5. Using a Different Model

You can use any GGUF model available in the [Models Hub](https://sparknlp.org/models) by changing `setModelName`. The model is automatically downloaded at runtime.

In [ ]:
# # Use a different pretrained model
# # (Uncomment and change the model name to try a different one)

# entity_extractor_custom = LLMEntityExtractor() \
#     .pretrained("phi3.5_mini_4k_instruct_q4_gguf") \
#     .setInputCols(["document"]) \
#     .setOutputCol("entities") \
#     .setEntityTypes(["PERSON", "ORGANIZATION", "LOCATION"]) \
#     .setNPredict(500) \
#     .setNGpuLayers(99) \
#     .setTemperature(0.1)

## 6. Case Sensitivity

By default, entity matching is **case-insensitive**. Set `setCaseSensitive(True)` if you need exact case matching between the LLM output and the source text.

In [25]:
# Case-sensitive matching example
case_sensitive_entity_extractor = LLMEntityExtractor() \
    .pretrained("qwen3_4b_bf16_gguf") \
    .setInputCols(["document"]) \
    .setOutputCol("entities") \
    .setEntityTypes(["PERSON", "ORGANIZATION", "ACRONYM"]) \
    .setCaseSensitive(True) \
    .setNPredict(500) \
    .setNGpuLayers(99) \
    .setTemperature(0.1)

cs_pipeline = Pipeline(stages=[document_assembler, case_sensitive_entity_extractor])

cs_data = spark.createDataFrame([
    ["The WHO and CDC issued guidelines. Dr. Alan Turing worked at the NPL."]
]).toDF("text")

cs_result = cs_pipeline.fit(cs_data).transform(cs_data)
cs_result.select(
    F.explode("entities").alias("entity")
).select(
    F.col("entity.result").alias("text"),
    F.col("entity.metadata.entity").alias("type"),
    F.col("entity.begin").alias("begin"),
    F.col("entity.end").alias("end")
).show(50, truncate=False)


qwen3_4b_bf16_gguf download started this may take some time.
Approximate size to download 6 GB
[OK!]
+---------------+------------+-----+---+
|text           |type        |begin|end|
+---------------+------------+-----+---+
|WHO            |ORGANIZATION|4    |6  |
|CDC            |ORGANIZATION|12   |14 |
|Dr. Alan Turing|PERSON      |35   |49 |
|NPL            |ORGANIZATION|65   |67 |
+---------------+------------+-----+---+



In [26]:
spark.stop()

For more information, see:
- [LLMEntityExtractor API Docs](https://sparknlp.org/api/com/johnsnowlabs/nlp/annotators/ner/dl/LLMEntityExtractor)
- [Spark NLP Models Hub](https://sparknlp.org/models)
- [Spark NLP Documentation](https://sparknlp.org/docs/en/annotators#llmentityextractor)